In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
from torchvision.datasets import CIFAR10

In [ ]:
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

trainset = CIFAR10(root="./data", train=True, download=True, transform=transform)
testset = CIFAR10(root="./data", train=False, download=True, transform=transform)

100%|██████████| 170M/170M [34:08<00:00, 83.2kB/s]


In [ ]:
trainset

Dataset CIFAR10
    Number of datapoints: 50000
    Root location: ./data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
           )

In [ ]:
trainloader = DataLoader(trainset, batch_size = 64, shuffle=True)
testloader = DataLoader(testset, batch_size = 64)

In [ ]:
class CNN(nn.Module):
  def __init__(self):
    super(CNN, self).__init__()

    self.conv_layers = nn.Sequential(
        nn.Conv2d(3,32, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2, 2),

        nn.Conv2d(32,64, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2, 2),

        nn.Conv2d(64,128, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2, 2)
    )

    self.fc_layers = nn.Sequential(
        nn.Linear(4*4*128, 256),
        nn.ReLU(),

        nn.Linear(256, 10)
    )

  def forward(self, x):
    x = self.conv_layers(x)
    x = x.view(x.size(0), -1)
    x = self.fc_layers(x)
    return x



In [ ]:
model = CNN()

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [ ]:
epochs = 10

for epoch in range(epochs):
  epoch_training_loss = 0.0

  for image, labels in trainloader:
    optimizer.zero_grad()

    output = model.forward(image)
    loss = criterion(output, labels)
    loss.backward()
    optimizer.step()

    epoch_training_loss += loss.item()

  print(f"epoch={epoch+1}/{epochs} & loss={epoch_training_loss/len(trainloader)}")

epoch=1/10 & loss=1.3579014358313188
epoch=2/10 & loss=0.9290113862975479
epoch=3/10 & loss=0.7492706837023005
epoch=4/10 & loss=0.6238634180839714
epoch=5/10 & loss=0.518721998766865
epoch=6/10 & loss=0.4319958807829091
epoch=7/10 & loss=0.3465396114970412
epoch=8/10 & loss=0.2720113176938213
epoch=9/10 & loss=0.21731865807624576
epoch=10/10 & loss=0.1647716640111278


In [ ]:
correct_labels = 0
total_labels = 0

model.eval()

with torch.no_grad():
  for image, labels in testloader:
    outputs = model.forward(image)
    _, prediction = torch.max(outputs, 1)

    correct_labels += (prediction == labels).sum().item()
    total_labels += labels.size(0)

print(f"accuracy = {correct_labels/total_labels * 100}")

accuracy = 74.8
